In [53]:
# !pip uninstall -y torch torchvision torchaudio torchtext


In [54]:
# from google.colab import drive
# drive.mount('/content/drive', force_remount=True)


In [55]:
# !pip install torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 --index-url https://download.pytorch.org/whl/cu121


In [56]:
!pip install torchtext==0.18.0


In [57]:
# !python -m spacy download en_core_web_sm
# !python -m spacy download fr_core_news_sm


In [58]:
# import torch
# import torchtext

# print("Torch:", torch.__version__)
# print("CUDA available:", torch.cuda.is_available())
# print("torchtext:", torchtext.__version__)


In [59]:

# Cell 1: Import & Config

# %%
import os
import math
import time
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

import torchtext
torchtext.disable_torchtext_deprecation_warning()
from torchtext.data.utils import get_tokenizer

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

nltk.download("punkt", quiet=True)

# ============ ĐƯỜNG DẪN DỮ LIỆU ============
# TODO: sửa đường dẫn này nếu khác
DATA_ROOT = r"/content/drive/MyDrive/NLP PROJECT/data/multi30k_en_fr"  # chứa train.en, train.fr, ...

# ============ ĐƯỜNG DẪN LƯU MODEL ============
MODEL_DIR = r"/content/drive/MyDrive/NLP PROJECT"
os.makedirs(MODEL_DIR, exist_ok=True)

# ============ HYPERPARAMETER ===============
MAX_VOCAB_SIZE = 10_000

EMB_DIM = 512          # tăng từ 256 -> 512
HID_DIM = 512
NUM_LAYERS = 2
DROPOUT = 0.4          # tăng từ 0.3 -> 0.4 cho regularization mạnh hơn

BATCH_SIZE = 64
NUM_EPOCHS = 25        # chạy nhiều epoch hơn (vd 25)
LEARNING_RATE = 5e-4   # giảm LR từ 1e-3 -> 5e-4
TEACHER_FORCING_RATIO = 0.5
CLIP = 1.0

EARLY_STOPPING_PATIENCE = 5

MAX_LEN_GENERATE = 50

# ============ THIẾT BỊ =====================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============ TOKEN ĐẶC BIỆT ===============
PAD_TOKEN = "<pad>"
SOS_TOKEN = "<sos>"
EOS_TOKEN = "<eos>"
UNK_TOKEN = "<unk>"

SPECIAL_TOKENS = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN]

print("DEVICE:", DEVICE)
print("MODEL_DIR:", MODEL_DIR)


DEVICE: cuda
MODEL_DIR: /content/drive/MyDrive/NLP PROJECT


In [60]:

# Cell 2: Data, Tokenizer, Vocab, Dataset, DataLoader
from torchtext.vocab import build_vocab_from_iterator   # <-- thêm dòng này

# %%
# 1. Tokenizer với spaCy (cần cài & download model spaCy trước)
#   pip install spacy torchtext
#   python -m spacy download en_core_web_sm
#   python -m spacy download fr_core_news_sm
en_tokenizer = get_tokenizer("spacy", language="en_core_web_sm")
fr_tokenizer = get_tokenizer("spacy", language="fr_core_news_sm")


def read_parallel_file(root, split):
    """
    Đọc song song split.en và split.fr
    Trả về list (en_sentence, fr_sentence)
    """
    en_path = os.path.join(root, f"{split}.en")
    fr_path = os.path.join(root, f"{split}.fr")

    assert os.path.exists(en_path), f"Không tìm thấy file: {en_path}"
    assert os.path.exists(fr_path), f"Không tìm thấy file: {fr_path}"

    pairs = []
    with open(en_path, "r", encoding="utf-8") as f_en, open(
        fr_path, "r", encoding="utf-8"
    ) as f_fr:
        for en_line, fr_line in zip(f_en, f_fr):
            en_line = en_line.strip()
            fr_line = fr_line.strip()
            if en_line == "" or fr_line == "":
                continue
            pairs.append((en_line, fr_line))
    return pairs


def yield_tokens_en(pairs):
    for en, _ in pairs:
        yield en_tokenizer(en)


def yield_tokens_fr(pairs):
    for _, fr in pairs:
        yield fr_tokenizer(fr)


def build_vocab(pairs):
    from collections import Counter
    from torchtext.vocab import build_vocab_from_iterator  # <-- thêm dòng này

    def build_with_limit(yield_tokens_fn):
        counter = Counter()
        for tokens in yield_tokens_fn(pairs):
            counter.update(tokens)

        most_common = [w for w, _ in counter.most_common(MAX_VOCAB_SIZE)]

        # iterator phải trả list token list
        def iterator():
            yield most_common

        vocab = build_vocab_from_iterator(
            iterator(),
            specials=SPECIAL_TOKENS,
            special_first=True,
        )

        vocab.set_default_index(vocab[UNK_TOKEN])
        return vocab

    src_vocab = build_with_limit(yield_tokens_en)
    trg_vocab = build_with_limit(yield_tokens_fr)
    return src_vocab, trg_vocab

class TranslationDataset(Dataset):
    def __init__(self, pairs, src_vocab, trg_vocab):
        self.data = []
        self.src_vocab = src_vocab
        self.trg_vocab = trg_vocab

        for en, fr in pairs:
            src_tokens = [SOS_TOKEN] + list(en_tokenizer(en)) + [EOS_TOKEN]
            trg_tokens = [SOS_TOKEN] + list(fr_tokenizer(fr)) + [EOS_TOKEN]

            src_ids = torch.tensor(
                [src_vocab[token] for token in src_tokens],
                dtype=torch.long,
            )
            trg_ids = torch.tensor(
                [trg_vocab[token] for token in trg_tokens],
                dtype=torch.long,
            )

            self.data.append((src_ids, trg_ids))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


def collate_fn(batch, pad_idx):
    """
    batch: list[(src_ids, trg_ids)]
    Trả về:
        src_padded: [B, src_len]
        src_lengths: [B]
        trg_padded: [B, trg_len]
        trg_lengths: [B]
    """
    src_list, trg_list = zip(*batch)

    src_lengths = torch.tensor([len(s) for s in src_list], dtype=torch.long)
    trg_lengths = torch.tensor([len(t) for t in trg_list], dtype=torch.long)

    # sort theo src_len giảm dần
    src_lengths, sort_idx = src_lengths.sort(descending=True)
    src_list = [src_list[i] for i in sort_idx]
    trg_list = [trg_list[i] for i in sort_idx]
    trg_lengths = torch.tensor([len(t) for t in trg_list], dtype=torch.long)

    src_padded = pad_sequence(src_list, batch_first=True, padding_value=pad_idx)
    trg_padded = pad_sequence(trg_list, batch_first=True, padding_value=pad_idx)

    return (
        src_padded.to(DEVICE),
        src_lengths.to(DEVICE),
        trg_padded.to(DEVICE),
        trg_lengths.to(DEVICE),
    )


def build_dataloaders():
    train_pairs = read_parallel_file(DATA_ROOT, "train")
    val_pairs = read_parallel_file(DATA_ROOT, "val")
    test_pairs = read_parallel_file(DATA_ROOT, "test")

    src_vocab, trg_vocab = build_vocab(train_pairs)

    pad_idx = src_vocab[PAD_TOKEN]

    train_dataset = TranslationDataset(train_pairs, src_vocab, trg_vocab)
    val_dataset = TranslationDataset(val_pairs, src_vocab, trg_vocab)
    test_dataset = TranslationDataset(test_pairs, src_vocab, trg_vocab)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=lambda b: collate_fn(b, pad_idx),
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=lambda b: collate_fn(b, pad_idx),
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=1,
        shuffle=False,
        collate_fn=lambda b: collate_fn(b, pad_idx),
    )

    return train_loader, val_loader, test_loader, src_vocab, trg_vocab, pad_idx


In [61]:

# Cell 3: Model với BiLSTM Encoder + Luong Attention (Luong-style)

# %%
import random

class Encoder(nn.Module):
    """
    BiLSTM Encoder
    - Mỗi hướng có hidden size = hid_dim // 2
    - Kết quả concat lại => output dim = hid_dim (tương thích decoder)
    """
    def __init__(self, input_dim, emb_dim, hid_dim, num_layers, dropout):
        super().__init__()
        self.hid_dim = hid_dim
        self.num_layers = num_layers

        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.dropout = nn.Dropout(dropout)

        # BiLSTM: mỗi hướng = hid_dim // 2 -> concat 2 hướng = hid_dim
        assert hid_dim % 2 == 0, "HID_DIM phải chẵn để chia 2 cho BiLSTM."
        self.rnn = nn.LSTM(
            emb_dim,
            hid_dim // 2,
            num_layers=num_layers,
            dropout=dropout,
            batch_first=True,
            bidirectional=True,
        )

    def _cat_directions(self, h):
        """
        h: [num_layers * 2, B, hid_dim // 2]
        Trả về: [num_layers, B, hid_dim] bằng cách concat forward + backward
        """
        num_layers_times_directions, batch_size, hid_half = h.size()
        assert num_layers_times_directions == self.num_layers * 2

        # [num_layers, 2, B, hid_half]
        h = h.view(self.num_layers, 2, batch_size, hid_half)
        # concat 2 hướng ở dim hidden: [num_layers, B, hid_dim]
        h = torch.cat((h[:, 0, :, :], h[:, 1, :, :]), dim=2)
        return h

    def forward(self, src, src_lengths):
        # src: [B, src_len]
        embedded = self.dropout(self.embedding(src))  # [B, src_len, emb_dim]

        packed = pack_padded_sequence(
            embedded, src_lengths.cpu(), batch_first=True, enforce_sorted=True
        )
        packed_outputs, (hidden, cell) = self.rnn(packed)
        outputs, _ = pad_packed_sequence(packed_outputs, batch_first=True)
        # outputs: [B, src_len, hid_dim] (2 * (hid_dim//2))

        # hidden, cell: [num_layers*2, B, hid_dim//2] -> [num_layers, B, hid_dim]
        hidden = self._cat_directions(hidden)
        cell = self._cat_directions(cell)

        return outputs, hidden, cell


class AttnDecoder(nn.Module):
    """
    Luong attention (dot) + LSTM Decoder (1 hướng, left-to-right)
    - encoder_outputs: [B, src_len, hid_dim]
    - decoder output:  [B, 1, hid_dim]
    """
    def __init__(self, output_dim, emb_dim, hid_dim, num_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.hid_dim = hid_dim
        self.num_layers = num_layers

        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.dropout = nn.Dropout(dropout)

        self.rnn = nn.LSTM(
            emb_dim,
            hid_dim,
            num_layers=num_layers,
            dropout=dropout,
            batch_first=True,
        )
        # concat [decoder_output; context] -> Linear
        self.fc_out = nn.Linear(hid_dim * 2, output_dim)

    def forward(self, input, hidden, cell, encoder_outputs):
        """
        input: [B]
        hidden, cell: [num_layers, B, hid_dim]
        encoder_outputs: [B, src_len, hid_dim]
        """
        input = input.unsqueeze(1)  # [B, 1]
        embedded = self.dropout(self.embedding(input))  # [B, 1, emb_dim]

        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        # output: [B, 1, hid_dim]

        # Luong dot attention: scores = output @ encoder_outputs^T
        # encoder_outputs: [B, src_len, hid_dim] -> [B, hid_dim, src_len]
        scores = torch.bmm(output, encoder_outputs.transpose(1, 2))  # [B, 1, src_len]
        attn_weights = F.softmax(scores, dim=-1)                     # [B, 1, src_len]

        # context: weighted sum of encoder_outputs
        context = torch.bmm(attn_weights, encoder_outputs)           # [B, 1, hid_dim]

        # concat context + decoder output
        concat = torch.cat((output, context), dim=2)                 # [B, 1, 2*hid_dim]
        prediction = self.fc_out(concat.squeeze(1))                  # [B, output_dim]

        return prediction, hidden, cell, attn_weights.squeeze(1)


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device=DEVICE):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

        assert encoder.hid_dim == decoder.hid_dim, \
            "Encoder HID_DIM và Decoder HID_DIM phải bằng nhau!"

    def forward(self, src, src_lengths, trg, teacher_forcing_ratio=0.5):
        """
        src: [B, src_len]
        trg: [B, trg_len]
        """
        batch_size = src.size(0)
        trg_len = trg.size(1)
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)

        encoder_outputs, hidden, cell = self.encoder(src, src_lengths)

        input = trg[:, 0]  # <sos>

        for t in range(1, trg_len):
            output, hidden, cell, _ = self.decoder(
                input, hidden, cell, encoder_outputs
            )
            outputs[:, t, :] = output

            top1 = output.argmax(1)

            if random.random() < teacher_forcing_ratio:
                input = trg[:, t]
            else:
                input = top1

        return outputs


In [62]:

# Cell 4: Train / Eval / Translate / BLEU

# %%
def epoch_time(start_time, end_time):
    elapsed = end_time - start_time
    mins = int(elapsed / 60)
    secs = int(elapsed - mins * 60)
    return mins, secs


def train_one_epoch(model, iterator, optimizer, criterion, pad_idx):
    model.train()
    epoch_loss = 0

    for src, src_len, trg, trg_len in tqdm(iterator, desc="Đang train", total=len(iterator)):
        optimizer.zero_grad()

        output = model(
            src,
            src_len,
            trg,
            teacher_forcing_ratio=TEACHER_FORCING_RATIO,
        )

        # output: [B, trg_len, output_dim]
        output_dim = output.shape[-1]
        output = output[:, 1:, :].contiguous().view(-1, output_dim)
        trg = trg[:, 1:].contiguous().view(-1)

        loss = criterion(output, trg)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(iterator)


def evaluate(model, iterator, criterion, pad_idx):
    model.eval()
    epoch_loss = 0

    with torch.no_grad():
        for src, src_len, trg, trg_len in tqdm(iterator, desc="Đang evaluate", total=len(iterator)):
            output = model(src, src_len, trg, teacher_forcing_ratio=0.0)

            output_dim = output.shape[-1]
            output = output[:, 1:, :].contiguous().view(-1, output_dim)
            trg = trg[:, 1:].contiguous().view(-1)

            loss = criterion(output, trg)
            epoch_loss += loss.item()

    return epoch_loss / len(iterator)

###############################################
# BEAM SEARCH DECODING CHO SEQ2SEQ + ATTENTION
###############################################

def beam_search_decode(model, encoder_outputs, hidden, cell, trg_vocab, beam_size=5, max_len=50):
    """
    Beam search decoding cho Seq2Seq + Attention.
    Trả về: danh sách token ID của câu được dịch.
    """

    EOS = trg_vocab[EOS_TOKEN]

    # mỗi phần tử beam = (score, seq, hidden, cell)
    beams = [(0.0, [trg_vocab[SOS_TOKEN]], hidden, cell)]  # score = log prob

    finished = []

    for _ in range(max_len):
        new_beams = []

        for score, seq, h, c in beams:
            last_token = seq[-1]

            if last_token == EOS:
                finished.append((score, seq))
                continue

            input_token = torch.tensor([last_token], dtype=torch.long, device=DEVICE)

            with torch.no_grad():
                output, new_h, new_c, _ = model.decoder(
                    input_token, h, c, encoder_outputs
                )

            # log-softmax để cộng log-prob
            log_probs = torch.log_softmax(output, dim=1)  # [1, vocab]

            topk_log_probs, topk_ids = torch.topk(log_probs, beam_size, dim=1)

            for k in range(beam_size):
                candidate_score = score + topk_log_probs[0][k].item()
                candidate_seq = seq + [topk_ids[0][k].item()]
                new_beams.append((candidate_score, candidate_seq, new_h, new_c))

        # giữ lại beam tốt nhất
        new_beams = sorted(new_beams, key=lambda x: x[0], reverse=True)[:beam_size]
        beams = new_beams

        if len(finished) >= beam_size:
            break

    if len(finished) == 0:
        finished = beams

    # chọn câu có điểm cao nhất
    best_score, best_seq = sorted(finished, key=lambda x: x[0], reverse=True)[0]
    return best_seq


def translate_sentence(
    sentence: str,
    model,
    src_vocab,
    trg_vocab,
    beam_size=5,
    max_len=50
):
    """
    Dịch 1 câu bằng beam search.
    """

    model.eval()

    # tokenize EN
    tokens_en = [SOS_TOKEN] + list(en_tokenizer(sentence.lower())) + [EOS_TOKEN]
    src_ids = [src_vocab[token] for token in tokens_en]

    src_tensor = torch.tensor(src_ids).unsqueeze(0).to(DEVICE)
    src_len = torch.tensor([len(src_ids)]).to(DEVICE)

    # Encoder
    with torch.no_grad():
        encoder_outputs, hidden, cell = model.encoder(src_tensor, src_len)

    # beam search
    best_seq = beam_search_decode(
        model, encoder_outputs, hidden, cell, trg_vocab,
        beam_size=beam_size,
        max_len=max_len
    )

    # convert tokens
    trg_tokens = [trg_vocab.lookup_token(idx) for idx in best_seq]

    # bỏ <sos>
    trg_tokens = trg_tokens[1:]
    if EOS_TOKEN in trg_tokens:
        trg_tokens = trg_tokens[:trg_tokens.index(EOS_TOKEN)]

    return " ".join(trg_tokens)



def compute_bleu(model, test_loader, src_vocab, trg_vocab, max_samples=100):
    smoothie = SmoothingFunction().method4
    model.eval()
    scores = []

    inv_trg_vocab = trg_vocab

    with torch.no_grad():
        for i, (src, src_len, trg, trg_len) in enumerate(test_loader):
            if i >= max_samples:
                break

            # target câu tiếng Pháp
            trg_ids = trg[0].tolist()
            trg_tokens = [inv_trg_vocab.lookup_token(idx) for idx in trg_ids]

            clean_trg = []
            for tok in trg_tokens:
                if tok in [SOS_TOKEN, EOS_TOKEN, PAD_TOKEN]:
                    continue
                clean_trg.append(tok)
            reference = [clean_trg]

            # source tiếng Anh
            src_ids = src[0].tolist()
            src_tokens = [src_vocab.lookup_token(idx) for idx in src_ids]
            clean_src_tokens = []
            for tok in src_tokens:
                if tok in [SOS_TOKEN, EOS_TOKEN, PAD_TOKEN]:
                    continue
                clean_src_tokens.append(tok)
            src_sentence = " ".join(clean_src_tokens)

            # dịch
            hypothesis_sentence = translate_sentence(
                src_sentence, model, src_vocab, trg_vocab
            )
            hypothesis_tokens = hypothesis_sentence.split()

            bleu = sentence_bleu(reference, hypothesis_tokens, smoothing_function=smoothie)
            scores.append(bleu)

    if len(scores) == 0:
        return 0.0
    return sum(scores) / len(scores)


In [63]:

# Cell 5: Train + Lưu model + Vẽ loss + Tính BLEU

# %%
def train_and_eval(resume=True):
    print("Thiết bị:", DEVICE)

    # 1) DataLoader + vocab (lần nào cũng build lại, nhưng vocab sẽ được overwrite từ checkpoint nếu resume)
    (
        train_loader,
        val_loader,
        test_loader,
        src_vocab,
        trg_vocab,
        pad_idx,
    ) = build_dataloaders()

    input_dim = len(src_vocab)
    output_dim = len(trg_vocab)

    print(f"Vocab EN size: {input_dim}, Vocab FR size: {output_dim}")

    # 2) Model
    encoder = Encoder(
        input_dim=input_dim,
        emb_dim=EMB_DIM,
        hid_dim=HID_DIM,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
    )
    decoder = AttnDecoder(
        output_dim=output_dim,
        emb_dim=EMB_DIM,
        hid_dim=HID_DIM,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
    )

    model = Seq2Seq(encoder, decoder, device=DEVICE).to(DEVICE)
    print(model)

    # 3) Loss, optimizer
    criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    # Đường dẫn lưu checkpoint
    best_model_path = os.path.join(MODEL_DIR, "best_model_suggestion.pth")
    training_state_path = os.path.join(MODEL_DIR, "last_checkpoint_suggestion.pth")

    # ====== PHẦN QUAN TRỌNG: LOAD TRẠNG THÁI NẾU MUỐN RESUME ======
    start_epoch = 0
    best_valid_loss = float("inf")
    epochs_no_improve = 0
    train_losses = []
    val_losses = []

    if resume and os.path.exists(training_state_path):
        print("🔁 Phát hiện last_checkpoint_suggestion.pth, đang load để train tiếp...")
        ckpt = torch.load(training_state_path, map_location=DEVICE)

        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])

        start_epoch       = ckpt["epoch"] + 1
        best_valid_loss   = ckpt["best_valid_loss"]
        epochs_no_improve = ckpt["epochs_no_improve"]
        train_losses      = ckpt["train_losses"]
        val_losses        = ckpt["val_losses"]

        # Overwrite vocab & pad_idx bằng đúng cái đã dùng trước đó
        src_vocab = ckpt["src_vocab"]
        trg_vocab = ckpt["trg_vocab"]
        pad_idx   = ckpt["pad_idx"]

        print(f"✅ Resume từ epoch {start_epoch}/{NUM_EPOCHS-1}")
    else:
        print("🚀 Không resume, train mới từ epoch 0.")

    # Nếu đã train đủ NUM_EPOCHS rồi thì khỏi train nữa
    if start_epoch >= NUM_EPOCHS:
        print(f"Đã train đủ {NUM_EPOCHS} epoch rồi, bỏ qua bước train.")
    else:
        # 4) Vòng lặp train (có resume)
        for epoch in range(start_epoch, NUM_EPOCHS):
            start_time = time.time()
            print(f"\n===== Epoch {epoch + 1}/{NUM_EPOCHS} =====")

            train_loss = train_one_epoch(model, train_loader, optimizer, criterion, pad_idx)
            valid_loss = evaluate(model, val_loader, criterion, pad_idx)

            train_losses.append(train_loss)
            val_losses.append(valid_loss)

            end_time = time.time()
            epoch_mins, epoch_secs = epoch_time(start_time, end_time)

            print(f"Epoch: {epoch+1:02} | Time: {epoch_mins}m {epoch_secs}s")
            print(f"\tTrain Loss: {train_loss:.3f} | PPL: {math.exp(train_loss):.3f}")
            print(f"\tVal   Loss: {valid_loss:.3f} | PPL: {math.exp(valid_loss):.3f}")

            # 4.1) Lưu best model (như cũ)
            if valid_loss < best_valid_loss:
                best_valid_loss = valid_loss
                epochs_no_improve = 0

                torch.save(
                    {
                        "model_state_dict": model.state_dict(),
                        "src_vocab": src_vocab,
                        "trg_vocab": trg_vocab,
                        "pad_idx": pad_idx,
                    },
                    best_model_path,
                )
                print(f"=> Lưu best model: {best_model_path}")
            else:
                epochs_no_improve += 1

            # 4.2) Luôn luôn lưu state để lần sau resume
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_valid_loss": best_valid_loss,
                    "epochs_no_improve": epochs_no_improve,
                    "train_losses": train_losses,
                    "val_losses": val_losses,
                    "src_vocab": src_vocab,
                    "trg_vocab": trg_vocab,
                    "pad_idx": pad_idx,
                },
                training_state_path,
            )
            print(f"💾 Đã lưu training_state tại epoch {epoch+1} -> {training_state_path}")

            if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
                print("Early stopping do val_loss không giảm nữa.")
                break

    # 5) Vẽ loss (nếu train_losses rỗng thì bỏ qua)
    if len(train_losses) > 0:
        plt.figure()
        plt.plot(train_losses, label="Train Loss")
        plt.plot(val_losses, label="Val Loss")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.legend()
        plt.title("Train/Val Loss")

        loss_fig_path = os.path.join(MODEL_DIR, "loss_curve_suggestion.png")
        plt.savefig(loss_fig_path)
        plt.close()
        print(f"=> Đã lưu loss_curve_suggestion.png tại: {loss_fig_path}")
    else:
        print("⚠ Không có dữ liệu loss để vẽ (có thể vì đã train đủ từ trước).")

    # 6) Load best model và tính BLEU (như cũ)
    print("Đánh giá BLEU trên tập test...")
    ckpt_best = torch.load(best_model_path, map_location=DEVICE)
    model.load_state_dict(ckpt_best["model_state_dict"])
    src_vocab = ckpt_best["src_vocab"]
    trg_vocab = ckpt_best["trg_vocab"]
    pad_idx   = ckpt_best["pad_idx"]

    bleu = compute_bleu(model, test_loader, src_vocab, trg_vocab, max_samples=100)
    print(f"BLEU trung bình (100 câu test đầu): {bleu:.4f}")

    return model, src_vocab, trg_vocab



In [64]:

# Cell 6: Gọi train_and_eval() và test dịch

# %%
model, src_vocab, trg_vocab = train_and_eval()

# Thử dịch 1 câu
test_sentence = "a man is riding a bicycle in the street"

print("EN:", test_sentence)
print("FR:", translate_sentence(
    test_sentence,
    model,
    src_vocab,
    trg_vocab,
    beam_size=5    # <--- Bạn có thể chỉnh 3, 5, 10 tuỳ ý
))


Thiết bị: cuda
Vocab EN size: 10004, Vocab FR size: 10004
Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(10004, 512)
    (dropout): Dropout(p=0.4, inplace=False)
    (rnn): LSTM(512, 256, num_layers=2, batch_first=True, dropout=0.4, bidirectional=True)
  )
  (decoder): AttnDecoder(
    (embedding): Embedding(10004, 512)
    (dropout): Dropout(p=0.4, inplace=False)
    (rnn): LSTM(512, 512, num_layers=2, batch_first=True, dropout=0.4)
    (fc_out): Linear(in_features=1024, out_features=10004, bias=True)
  )
)
🚀 Không resume, train mới từ epoch 0.

===== Epoch 1/25 =====


Đang train:   0%|          | 0/454 [00:00<?, ?it/s]

Đang evaluate:   0%|          | 0/16 [00:00<?, ?it/s]

Epoch: 01 | Time: 1m 36s
	Train Loss: 4.411 | PPL: 82.367
	Val   Loss: 3.853 | PPL: 47.114
=> Lưu best model: /content/drive/MyDrive/NLP PROJECT/best_model_suggestion.pth
💾 Đã lưu training_state tại epoch 1 -> /content/drive/MyDrive/NLP PROJECT/last_checkpoint_suggestion.pth

===== Epoch 2/25 =====


Đang train:   0%|          | 0/454 [00:00<?, ?it/s]

Đang evaluate:   0%|          | 0/16 [00:00<?, ?it/s]

... (36 outputs left out)

Epoch: 14 | Time: 1m 39s
	Train Loss: 1.134 | PPL: 3.108
	Val   Loss: 3.008 | PPL: 20.249
💾 Đã lưu training_state tại epoch 14 -> /content/drive/MyDrive/NLP PROJECT/last_checkpoint_suggestion.pth

===== Epoch 15/25 =====


Đang train:   0%|          | 0/454 [00:00<?, ?it/s]

Đang evaluate:   0%|          | 0/16 [00:00<?, ?it/s]

Epoch: 15 | Time: 1m 39s
	Train Loss: 1.092 | PPL: 2.981
	Val   Loss: 3.003 | PPL: 20.154
💾 Đã lưu training_state tại epoch 15 -> /content/drive/MyDrive/NLP PROJECT/last_checkpoint_suggestion.pth
Early stopping do val_loss không giảm nữa.
=> Đã lưu loss_curve_suggestion.png tại: /content/drive/MyDrive/NLP PROJECT/loss_curve_suggestion.png
Đánh giá BLEU trên tập test...
BLEU trung bình (100 câu test đầu): 0.4740
EN: a man is riding a bicycle in the street
FR: Un homme fait du vélo dans la rue


In [ ]:
import torch
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.backends.cudnn.version())

12.1
True
8801


In [ ]:
print("hello") 